# Spike 04 — PageIndex Reasoning Graph

**Goal:** Feed the OCR'd Markdown into PageIndex and build a reasoning graph (tree structure) over the document.

**Time box:** 1-2 hours

**Question to answer:** Does PageIndex successfully index the bilingual OCR output and return a usable reasoning structure?

**Done when:** Documents are indexed in PageIndex and we can query them without errors.

---

### What is Vector-Less Reasoning RAG?

Traditional RAG:
```
Document → chunk into pieces → embed as vectors → store in vector DB
Query    → embed query       → find nearest chunks → send to LLM
```
Problem: chunking breaks tables and cross-references. Arabic embeddings are weaker than English ones.

PageIndex approach:
```
Document → build reasoning graph (understands structure)
Query    → reasoning traversal over graph → find relevant pages
```
This is why it suits our bilingual, table-heavy urban reports.

### API Key Setup
1. Go to [pageindex.ai](https://pageindex.ai)
2. Click "Try Now" or "Book a Demo"
3. Mention you're in a hackathon — many services give free credits for this
4. Add to `.env`: `PAGEINDEX_API_KEY=your_key_here`

---
### ⚠️ Important
PageIndex's API details aren't fully public. This notebook uses the most likely API shape  
based on their documentation. **Adjust endpoint URLs and payload keys** based on what  
their team sends you after signup.

## Step 1 — Setup

> **How PageIndex's `/markdown` endpoint actually works (confirmed from working code):**
> - Auth: `Authorization: Bearer <key>` in the header
> - Endpoint: `POST https://api.pageindex.ai/markdown` — **no trailing slash**
> - Upload: `multipart/form-data` with field `file`
> - Response: **synchronous** — returns `{"success": true, "doc_name": "...", "structure": [...nodes...]}` immediately
> - The `structure` is the full reasoning graph — no polling, no doc_id, no server-side retrieval
>
> The SDK's `submit_document()` / `is_retrieval_ready()` / `submit_query()` chain applies only to **PDF** uploads via `/doc/`.  
> For markdown: upload → get structure → search locally.

In [1]:
import requests
from dotenv import load_dotenv
from pathlib import Path
import os, json

load_dotenv(dotenv_path="../.env")

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
if not PAGEINDEX_API_KEY:
    raise ValueError("PAGEINDEX_API_KEY not found in .env")

BASE_URL = "https://api.pageindex.ai"

# Paths
AR_DIR   = Path("../ocr_output/arabic")
EN_DIR   = Path("../ocr_output/english")
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

ar_files = sorted(AR_DIR.glob("*.md"))
en_files = sorted(EN_DIR.glob("*.md"))

print(f"✅ Setup ready")
print(f"   English pages : {len(en_files)}")
print(f"   Arabic pages  : {len(ar_files)}")

✅ Setup ready
   English pages : 14
   Arabic pages  : 14


## Step 2 — Combine Pages into Single Documents

`submit_document()` takes a **file path** — one file per document.  
We combine all OCR pages into two files: one English, one Arabic.  
Each page is separated by a clear heading so PageIndex can distinguish them.

In [2]:
def combine_pages(md_files: list, output_path: Path, lang: str) -> Path:
    """Merge all page .md files into one document with page separators."""
    parts = []
    for f in md_files:
        content = f.read_text(encoding="utf-8").strip()
        parts.append(f"<!-- PAGE: {f.stem} -->\n\n{content}")
    combined = "\n\n---\n\n".join(parts)
    output_path.write_text(combined, encoding="utf-8")
    size_kb = output_path.stat().st_size // 1024
    print(f"✅ {lang}: {len(md_files)} pages → {output_path.name} ({size_kb} KB)")
    return output_path

en_combined = combine_pages(en_files, DATA_DIR / "tranquil_en_combined.md", "English")
ar_combined = combine_pages(ar_files, DATA_DIR / "tranquil_ar_combined.md", "Arabic")

✅ English: 14 pages → tranquil_en_combined.md (41 KB)
✅ Arabic: 14 pages → tranquil_ar_combined.md (41 KB)


## Step 3 — Upload to PageIndex and Get Reasoning Structure

`POST /markdown` processes the combined `.md` file synchronously and returns the full reasoning graph in the response body.  
We cache the structure to disk so we don't have to re-upload on every kernel restart.

In [3]:
STRUCTURE_CACHE = DATA_DIR / "pageindex_structures.json"

def upload_markdown(file_path: Path, label: str) -> list:
    """
    Upload a .md file to /markdown and return the list of structure nodes.
    Auth: Authorization: Bearer <key>  (NOT api_key header)
    No trailing slash on the endpoint.
    Response is synchronous — structure comes back immediately.
    """
    print(f"Uploading {label} ({file_path.name}, {file_path.stat().st_size // 1024} KB)...")
    with open(file_path, "rb") as f:
        response = requests.post(
            f"{BASE_URL}/markdown",
            headers={"Authorization": f"Bearer {PAGEINDEX_API_KEY}"},
            files={"file": (file_path.name, f, "text/markdown")},
            timeout=120
        )

    print(f"  HTTP {response.status_code}")
    if response.status_code != 200:
        print(f"  ❌ Error: {response.text[:300]}")
        return []

    result = response.json()

    if not result.get("success"):
        print(f"  ❌ Upload failed: {result}")
        return []

    structure = result.get("structure", [])
    doc_name  = result.get("doc_name", "unknown")
    print(f"  ✅ {label}: doc_name='{doc_name}', {len(structure)} nodes returned")
    return structure


if STRUCTURE_CACHE.exists():
    structures = json.loads(STRUCTURE_CACHE.read_text(encoding="utf-8"))
    print(f"⏭  Loaded from cache:")
    for lang, nodes in structures.items():
        print(f"   {lang}: {len(nodes)} nodes")
else:
    structures = {}

    en_structure = upload_markdown(en_combined, "English")
    ar_structure = upload_markdown(ar_combined, "Arabic")

    if en_structure:
        structures["english"] = en_structure
    if ar_structure:
        structures["arabic"] = ar_structure

    STRUCTURE_CACHE.write_text(json.dumps(structures, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\nSaved to {STRUCTURE_CACHE}")

print(f"\nTotal nodes — English: {len(structures.get('english', []))}, Arabic: {len(structures.get('arabic', []))}")

Uploading English (tranquil_en_combined.md, 41 KB)...
  HTTP 200
  ✅ English: doc_name='temp_tranquil_en_combined', 13 nodes returned
Uploading Arabic (tranquil_ar_combined.md, 41 KB)...
  HTTP 200
  ✅ Arabic: doc_name='temp_tranquil_ar_combined', 16 nodes returned

Saved to ..\data\pageindex_structures.json

Total nodes — English: 13, Arabic: 16


## Step 4 — Inspect the Reasoning Tree

PageIndex returns a list of **nodes**. Each node has:
- `node_id` — position in the tree (e.g. `"0000"`, `"0001"`)
- `title` — the section heading PageIndex detected
- `summary` — a condensed summary of this section
- `text` — the raw text from this section of the document
- `line_num` — source line number in the original file

In [4]:
for lang, nodes in structures.items():
    print(f"\n{'='*60}")
    print(f"REASONING TREE — {lang.upper()}")
    print(f"{'='*60}")
    print(f"Total nodes: {len(nodes)}")
    for node in nodes[:5]:
        node_id = node.get("node_id", "?")
        title   = node.get("title", "(no title)")
        summary = (node.get("summary") or node.get("text", ""))[:120]
        print(f"  [{node_id}] {title}")
        if summary:
            print(f"         {summary.replace(chr(10), ' ')}...")
    if len(nodes) > 5:
        print(f"  ... and {len(nodes)-5} more nodes")


REASONING TREE — ENGLISH
Total nodes: 13
  [0000] Opening Statement
         The partial document is an opening statement from the Madinah Development Authority, outlining its commitment to enhanci...
  [0001] TABLE OF CONTENTS
         The partial document is a table of contents for a report or book focused on urban development and livability, specifical...
  [0002] Executive Summary
         The partial document is an executive summary analyzing the livability of Al Madinah City, highlighting its strengths and...
  [0003] Population
         # Population...
  [0006] Al Madinah Today
         # Al Madinah Today...
  ... and 8 more nodes

REASONING TREE — ARABIC
Total nodes: 16
  [0000] الكلمة الافتتاحية
         يتناول الجزء المقدم من الوثيقة الكلمة الافتتاحية لتقرير صادر عن هيئة تطوير منطقة المدينة المنورة، حيث يبرز التزام الهيئة...
  [0001] المحتوى
         # المحتوى...
  [0014] الملخص التنفيذي
         # الملخص التنفيذي  يقدم هذا التقرير تحليلاً شاملاً لمستوى قابلية العيش في المدي

## Step 5 — Local Reasoning Search

The `/markdown` endpoint is synchronous — the structure IS the index.  
We search it locally by scoring each node against the query using keyword overlap.  
This is the retrieval step that replaces vector similarity.

In [5]:
def search_structure(query: str, nodes: list, top_k: int = 5) -> list:
    """
    Score each node by keyword overlap with the query.
    Returns top_k nodes sorted by descending relevance score.
    """
    query_words = set(query.lower().split())
    scored = []

    for node in nodes:
        combined = " ".join([
            node.get("title", ""),
            node.get("summary", ""),
            node.get("text", "")
        ]).lower()

        score = sum(1 for word in query_words if word in combined)
        if score > 0:
            scored.append((score, node))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [node for _, node in scored[:top_k]]


def retrieve(query: str, lang: str = "english", top_k: int = 5) -> list:
    """Retrieve top_k nodes from the given language structure."""
    nodes = structures.get(lang, [])
    results = search_structure(query, nodes, top_k)
    print(f"Retrieved {len(results)} nodes from {lang} structure for: '{query}'")
    return results


print("✅ search_structure() and retrieve() ready")

✅ search_structure() and retrieve() ready


## Step 6 — Test Queries

Run two queries — one English, one Arabic — and show the top matching nodes.  
This confirms the reasoning graph is searchable and relevant sections are retrieved correctly.

In [6]:
print("=" * 60)
print("ENGLISH QUERY")
print("=" * 60)

q_en = "What are the livability indicators for Madinah in 2024?"
en_results = retrieve(q_en, lang="english", top_k=3)

for i, node in enumerate(en_results):
    print(f"\nResult {i+1} — [{node.get('node_id')}] {node.get('title')}")
    print(f"  {node.get('text', '')[:400].replace(chr(10), ' ')}...")

ENGLISH QUERY
Retrieved 3 nodes from english structure for: 'What are the livability indicators for Madinah in 2024?'

Result 1 — [0002] Executive Summary
  # Executive Summary  This report comprehensively analyzes the livability of Al Madinah City and identifies strengths and areas for improvement through effective urban policies. Enhancing the city's livability can make it a role model in sustainability, resilience, inclusiveness, and innovation, achieving an ideal balance between society, environment, and economy.  **Spatial and Historical Perspect...

Result 2 — [0034] Preliminary Analysis of the Al Madinah Livable City Index
  # Preliminary Analysis of the Al Madinah Livable City Index  For the **Spatial and Historical Perspective** sector, Al Madinah City scores relatively well (81.0%) as the Global Beacon for Islamic and Cultural Enlightenment. But it would score better by continuing developing the cultural and heritage sites. Al Madinah City is Where tradition meets modernity w

In [7]:
print("=" * 60)
print("ARABIC QUERY")
print("=" * 60)

q_ar = "ما هي مؤشرات قابلية العيش في المدينة المنورة؟"
ar_results = retrieve(q_ar, lang="arabic", top_k=3)

for i, node in enumerate(ar_results):
    print(f"\nResult {i+1} — [{node.get('node_id')}] {node.get('title')}")
    print(f"  {node.get('text', '')[:400].replace(chr(10), ' ')}...")

ARABIC QUERY
Retrieved 3 nodes from arabic structure for: 'ما هي مؤشرات قابلية العيش في المدينة المنورة؟'

Result 1 — [0000] الكلمة الافتتاحية
  # الكلمة الافتتاحية  تسعى هيئة تطوير منطقة المدينة المنورة إلى تحسين قابلية العيش في المدينة المنورة وتعزيز جودة حياة سكانها وأن تكون ضمن أفضل مدن العالم القابلة للعيش، وذلك تماشياً مع أهداف برنامج جودة الحياة، أحد برامج رؤية المملكة 2030. ومن هذا المنطلق يسعدني أن أقدم لكم هذا التقرير المبني على بيانات موثوقة حسب المعايير العالمية حول قابلية العيش في مدينة المدينة المنورة، والذي يعكس الجهود المبذ...

Result 2 — [0058] تحليل مؤشر قابلية العيش للمدينة المنورة
  # تحليل مؤشر قابلية العيش للمدينة المنورة  حقق قطاع المنظور المكاني والتاريخي نتائج جيدة (81.0%) باعتبارها المنارة العالمية للإسلام والثقافة، وستكون النتائج أفضل مع استقرار تطوير المواقع التاريخية والتراثية، وتهيئتها للسكان والزوار، وكذلك زيادة الفعاليات الثقافية. وتعتبر المدينة المنورة المكان الذي تلتقي فيه العراقة بالحداثة مع تعدد المراكز الحضرية الناشئة وتطوير الحدائق العامة والشوارع 

## Step 7 — Spike Summary

In [8]:
en_nodes = structures.get("english", [])
ar_nodes = structures.get("arabic", [])

print("=" * 55)
print("SPIKE 04 — RESULTS")
print("=" * 55)
print(f"English nodes  : {len(en_nodes)}")
print(f"Arabic nodes   : {len(ar_nodes)}")
print(f"EN query hits  : {len(en_results)}")
print(f"AR query hits  : {len(ar_results)}")
print()

passed = len(en_nodes) > 0 and len(ar_nodes) > 0 and len(en_results) > 0 and len(ar_results) > 0

if passed:
    print("SPIKE PASSED")
    print("  PageIndex /markdown endpoint indexed both English and Arabic OCR output.")
    print("  Reasoning tree nodes are searchable — relevant sections retrieved for both languages.")
    print("  Next: spike_05_retrieval.ipynb - evaluate full answer quality with LLM generation")
else:
    print("SPIKE INCOMPLETE - check errors above")
    if not en_nodes:
        print("  English structure is empty")
    if not ar_nodes:
        print("  Arabic structure is empty")

SPIKE 04 — RESULTS
English nodes  : 13
Arabic nodes   : 16
EN query hits  : 3
AR query hits  : 3

SPIKE PASSED
  PageIndex /markdown endpoint indexed both English and Arabic OCR output.
  Reasoning tree nodes are searchable — relevant sections retrieved for both languages.
  Next: spike_05_retrieval.ipynb - evaluate full answer quality with LLM generation
